In [1]:
"""
ABIDES-Gym을 사용한 멀티 봇 시뮬레이션
다양한 전략을 가진 봇들의 시뮬레이션을 병렬로 실행하는 코드
"""

# 필요한 라이브러리 임포트
import gym
import numpy as np
import pandas as pd
from multiprocessing import Pool, cpu_count
from collections.abc import MutableMapping
import abides_gym

In [2]:
# 전략 클래스들 정의
class PolicyPassive:
    """항상 passive 액션(1)을 반환하는 전략"""
    def __init__(self):
        self.name = 'passive'
        
    def get_action(self, state):
        return 1
        

class PolicyAggressive:
    """항상 aggressive 액션(0)을 반환하는 전략"""
    def __init__(self):
        self.name = 'aggressive'
        
    def get_action(self, state):
        return 0
    

class PolicyRandom:
    """0과 1 중 무작위로 액션을 선택하는 전략"""
    def __init__(self):
        self.name = 'random'
        
    def get_action(self, state):
        return np.random.choice([0, 1])
    

class PolicyRandomWithNoAction:
    """0, 1, 2 중 무작위로 액션을 선택하는 전략 (2는 no action)"""
    def __init__(self):
        self.name = 'random_no_action'
        
    def get_action(self, state):
        return np.random.choice([0, 1, 2])

In [3]:
def generate_env(seed):
    """
    특정 파라미터로 환경을 생성하고 시드를 설정합니다.
    
    Args:
        seed (int): 환경의 랜덤 시드 값
        
    Returns:
        gym.Env: 설정된 시장 실행 환경
    """
    env = gym.make(
        "markets-execution-v0",
        background_config="rmsc04",
        timestep_duration="10S",
        execution_window="04:00:00",
        parent_order_size=20000,
        order_fixed_size=50,
        not_enough_reward_update=-100
    )
    env.seed(seed)
    return env


def flatten_dict(d: MutableMapping, sep: str = '.') -> MutableMapping:
    """
    중첩된 딕셔너리를 평탄화합니다.
    
    Args:
        d (MutableMapping): 평탄화할 중첩 딕셔너리
        sep (str): 중첩 키를 구분할 구분자, 기본값은 '.'
        
    Returns:
        MutableMapping: 평탄화된 딕셔너리
    """
    [flat_dict] = pd.json_normalize(d, sep=sep).to_dict(orient='records')
    return flat_dict


def run_episode(seed=None, policy=None):
    """
    주어진 시드와 정책으로 하나의 에피소드를 완전히 실행합니다.
    
    Args:
        seed (int, optional): 환경의 랜덤 시드 값
        policy (object): 행동을 결정하는 정책 객체, get_action 메서드가 있어야 함
        
    Returns:
        dict: 에피소드 실행 결과 정보가 담긴 딕셔너리 (보상, 정책 이름 등 포함)
    """
    env = generate_env(seed)
    state = env.reset()
    done = False
    episode_reward = 0 
    
    while not done:
        action = policy.get_action(state)
        state, reward, done, info = env.step(action)
        episode_reward += reward
    
    output = flatten_dict(info) 
    output['episode_reward'] = episode_reward
    output['name'] = policy.name
    return output


def wrap_run_episode(param):
    """
    run_episode 함수를 래핑하여 딕셔너리 파라미터를 언패킹합니다.
    
    Args:
        param (dict): run_episode에 전달할 파라미터 딕셔너리
        
    Returns:
        dict: run_episode 함수의 결과
    """
    return run_episode(**param)


def run_N_episode(N):
    """
    여러 정책에 대해 N개의 에피소드를 병렬로 실행합니다.
    
    Args:
        N (int): 각 정책별로 실행할 에피소드 수
        
    Returns:
        list: 모든 에피소드 실행 결과가 담긴 리스트
    """
    # 정책 정의
    policies = [
        PolicyAggressive(), 
        PolicyRandom(), 
        PolicyPassive(), 
        PolicyRandomWithNoAction()
    ]
    seeds = list(range(N))
    
    # 모든 정책과 시드 조합 생성
    tests = [{"policy": policy, 'seed': seed} for policy in policies for seed in seeds]
    
    total_tasks = len(tests)
    print(f"총 {total_tasks}개의 에피소드를 실행합니다...")
    
    # 키보드 인터럽트 처리를 위한 설정
    try:
        # 프로세스 수를 CPU 코어 수의 70%로 제한하여 시스템 부하 방지
        num_processes = max(1, int(cpu_count() * 0.70))
        print(f"{num_processes}개의 프로세스를 사용합니다.")
        
        outputs = []
        with Pool(processes=num_processes) as pool:
            # imap을 사용하여 결과가 도착하는 대로 처리
            for i, result in enumerate(pool.imap_unordered(
                wrap_run_episode, 
                tests, 
                chunksize=max(1, len(tests) // num_processes)
            )):
                outputs.append(result)
                # 진행 상황 표시
                if (i + 1) % max(1, total_tasks // 20) == 0 or (i + 1) == total_tasks:
                    print(f"진행률: {(i + 1) / total_tasks * 100:.1f}% ({i + 1}/{total_tasks})")
    except KeyboardInterrupt:
        print("병렬 처리가 사용자에 의해 중단되었습니다.")
        return outputs  # 중단 시점까지 수집된 결과 반환
    
    print("모든 에피소드 실행 완료!")
    return outputs

In [4]:
N = 5  # 각 정책별로 실행할 에피소드 수
outputs = run_N_episode(N)

총 20개의 에피소드를 실행합니다...
22개의 프로세스를 사용합니다.
진행률: 5.0% (1/20)
진행률: 10.0% (2/20)
진행률: 15.0% (3/20)
진행률: 20.0% (4/20)
진행률: 25.0% (5/20)
진행률: 30.0% (6/20)
진행률: 35.0% (7/20)
진행률: 40.0% (8/20)
진행률: 45.0% (9/20)
진행률: 50.0% (10/20)
진행률: 55.0% (11/20)
진행률: 60.0% (12/20)
진행률: 65.0% (13/20)
진행률: 70.0% (14/20)
진행률: 75.0% (15/20)
진행률: 80.0% (16/20)
진행률: 85.0% (17/20)
진행률: 90.0% (18/20)
진행률: 95.0% (19/20)
진행률: 100.0% (20/20)
모든 에피소드 실행 완료!


In [5]:
df = pd.DataFrame(outputs)
print(df.groupby('name')['episode_reward'].mean())

name
aggressive          23.616520
passive             28.744115
random               8.959740
random_no_action    41.631265
Name: episode_reward, dtype: float64
